# CiteCheck: fine-tune the verifier on a Colab GPU

**Runtime → Change runtime type → T4 GPU** before running anything.

Every number comes out of `eval.run_eval`, the same harness as Session 5, so runs are directly comparable with
`zero_shot_nli` (dev macro-F1 0.603, false-SUPPORT 0.108). Rules that still apply here:

* training reads `claims_train.jsonl` only; the 200 claims tau is tuned on are **held out** (run_eval refuses otherwise);
* dev is for reporting only: never pick hyperparameters by looking at dev numbers you then report as final;
* commit `eval/results/<run>/`, never `models/` (weights go to Drive or the Hugging Face Hub).

In [ ]:
# 1. settings: point these at your fork and branch
FORK_URL = "https://github.com/Lord2709/citecheck"   # <- your fork
BRANCH   = "feat/modeling-finetune-v2"
SAVE_TO_DRIVE = True                                 # keep checkpoints across Colab sessions
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. code + dependencies + data  (~3 min)
import os
if not os.path.exists("citecheck"):
    !git clone -b {BRANCH} {FORK_URL} citecheck
%cd citecheck
!git pull -q
!pip install -q -r requirements.txt
!python -m data.fetch_scifact

In [ ]:
# 3. where checkpoints live
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS = "/content/drive/MyDrive/citecheck-models"
else:
    MODELS = "models"
os.makedirs(MODELS, exist_ok=True)
print(MODELS)

## Experiment A: fine-tune the deployed base model (~10-15 min on a T4)

Same model the app runs today (`DeBERTa-v3-base-mnli-fever-anli`), now trained on SciFact train. Watch the
*held-out pair macro-F1* printed after each epoch: the best epoch is the one that is saved.

In [ ]:
!python -m eval.finetune_nli --out {MODELS}/ft-base --epochs 3 --batch-size 16 --lr 2e-5 && \
  python -m eval.run_eval --name finetuned_nli --retriever bm25 --verifier nli --model {MODELS}/ft-base \
    --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/lexical

## Experiment B: bigger zero-shot model, no training (~5 min)

Without this row we can't tell whether a gain in C comes from training or just from model size.
Before reporting it, check the model's license on its Hugging Face card and add it to `data/README.md`.

In [ ]:
LARGE = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
if not os.path.exists("eval/results/zero_shot_nli_large/results.json"):   # already run? skip it
  !python -m eval.run_eval --name zero_shot_nli_large --retriever bm25 --verifier nli --model {LARGE} \
    --batch-size 8 --oracle --compare-with eval/results/zero_shot_nli

## Experiment C: fine-tune the large model (~35-50 min on a T4)

Batch 4 x 4 accumulation = effective batch 16; gradient checkpointing keeps it inside 15 GB.
If Colab disconnects, the checkpoint of the best finished epoch is already on Drive.

In [ ]:
!python -m eval.finetune_nli --base {LARGE} --out {MODELS}/ft-large --epochs 2 --batch-size 4 --grad-accum 4 \
    --lr 6e-6 --gradient-checkpointing && \
  python -m eval.run_eval --name finetuned_nli_large --retriever bm25 --verifier nli --model {MODELS}/ft-large \
    --batch-size 8 --oracle --compare-with eval/results/zero_shot_nli --compare-with eval/results/zero_shot_nli_large

## Optional ablations (one change at a time, each its own run name)

* `--no-match-inference`: train on gold rationales only (the Session 5 recipe) to measure what the inference-style pairs add
* `--gold-doc-neutral 1`: add non-rationale sentences of gold papers as neutral (may cut false-SUPPORT, may cost recall)
* `--neg-per-claim 2`: more retrieved hard negatives

In [ ]:
# 4. results: the table of every run, and what to bring back to the repo
print(open("eval/results/LATEST.md").read())
import json
for run in ["zero_shot_nli", "finetuned_nli", "zero_shot_nli_large", "finetuned_nli_large"]:
    p = f"eval/results/{run}/results.json"
    if os.path.exists(p):
        r = json.load(open(p)); cl = r["claim_level"]
        print(f"{run:22s} macro-F1 {cl['macro_f1']:.3f} {[round(x, 3) for x in cl['macro_f1_ci95']]}  "
              f"false-SUPPORT {cl['false_support_rate']:.3f}  tau {r['config']['tau']}  "
              + "  ".join(f"vs {c['vs']} {c['diff']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}]" for c in r["paired_comparisons"]))

In [ ]:
# 5. download the results; each fine-tuned run's train_log.json is already in eval/results/<run>/
!zip -qr results.zip eval/results
from google.colab import files
files.download("results.zip")


## Before shipping a model

Only promote (`--promote`) if the paired CI vs `zero_shot_nli` is above zero **and** false-SUPPORT stays under the
0.15 ceiling in `docs/decision_rules.json`. The app then needs the checkpoint: upload it to the Hugging Face Hub
and promote the run with `--model <hub-id>` so `deployed_config.json` points at something the app can download.
If the fine-tuned model doesn't win, that is a result too: report it.